# exp_3a — Experiment 3A: CNN for Binary Image Classification (Cats vs Dogs)

Source-based notebook converted from the uploaded experiment code document.


In [ ]:
import zipfile
import os
import shutil

In [ ]:
zip_file = r"D:\DL DATASET\dogs_vs_cats_subset (1) (1).zip"

extract_path = r"D:\DL DATASET\dogs_vs_cats_subset"

In [ ]:
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

In [ ]:
for root, dirs, files in os.walk(extract_path):
    level = root.replace(extract_path, "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}{os.path.basename(root)}/")

In [ ]:
for root, dirs, files in os.walk(extract_path):
    if files:
        print("\nFolder:", root)
        print("Sample files:", files[:5])

In [ ]:
import os
import shutil

base_path = r"D:\DL DATASET\dogs_vs_cats_subset\dogs_vs_cats_subset"

for folder in ["train", "val"]:

    folder_path = os.path.join(base_path, folder)

    os.makedirs(os.path.join(folder_path, "cats"), exist_ok=True)
    os.makedirs(os.path.join(folder_path, "dogs"), exist_ok=True)

    for file in os.listdir(folder_path):

        src = os.path.join(folder_path, file)

        if not os.path.isfile(src):
            continue

        if file.startswith("cat"):
            shutil.move(src, os.path.join(folder_path, "cats", file))

        elif file.startswith("dog"):
            shutil.move(src, os.path.join(folder_path, "dogs", file))

print("Train and Validation folders organized successfully!")

In [ ]:
base_path = r"D:\DL DATASET\dogs_vs_cats_subset\dogs_vs_cats_subset"

for root, dirs, files in os.walk(base_path):
    if files:
        print(root)
        print(files[:5])

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout
)

In [ ]:
train_dir = r"D:\DL DATASET\dogs_vs_cats_subset\dogs_vs_cats_subset\train"

val_dir = r"D:\DL DATASET\dogs_vs_cats_subset\dogs_vs_cats_subset\val"

In [ ]:
IMG_SIZE = (150, 150)
BATCH_SIZE = 32

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

In [ ]:
val_datagen = ImageDataGenerator(
    rescale=1./255
)

In [ ]:
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=True
)

In [ ]:
val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

In [ ]:
import os

val_path = r"D:\DL DATASET\dogs_vs_cats_subset\dogs_vs_cats_subset\val"

count = 0
for root, dirs, files in os.walk(val_path):
    count += len([f for f in files if f.endswith(".jpg")])

print("Total Validation Images:", count)

In [ ]:
print(train_generator.class_indices)

In [ ]:
images,labels=next(train_generator)
plt.figure(figsize=(10,10))
for i in range(9):
    plt.subplot(3,3,i+1)
    plt.imshow(images[i])
    plt.title("Dog" if labels[i]==1 else "Cat")
    plt.axis("off")
    plt.show()

In [ ]:
model=Sequential([
tf.keras.Input(shape=(150,150,3)),
Conv2D(32,(3,3),activation="relu"),
MaxPooling2D(),
Conv2D(64,(3,3),activation="relu"),
MaxPooling2D(),
Conv2D(128,(3,3),activation="relu"),
MaxPooling2D(),
Flatten(),

Dense(128,activation="relu"),
Dropout(0.5),
Dense(1,activation="sigmoid")
])

In [ ]:
model.compile(
optimizer="adam",
loss="binary_crossentropy",
metrics=["accuracy"]
)
model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=15,
    callbacks=[early_stop]
)

In [ ]:
loss,acc=model.evaluate(val_generator)
print("Loss :",loss)
print("Accuracy :",acc)

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history.history["loss"],label="Training")
plt.plot(history.history["val_loss"],label="Validation")
plt.legend()
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
val_generator.reset()
pred=model.predict(val_generator)
pred=(pred>0.5).astype(int)
cm=confusion_matrix(val_generator.classes,pred)
print(cm)
print(classification_report(
val_generator.classes,
pred,
target_names=["Cat","Dog"]
))

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing import image

test_path = r"D:\DL DATASET\dogs_vs_cats_subset\dogs_vs_cats_subset\test"

files = os.listdir(test_path)

for f in files[:10]:

    img = image.load_img(
        os.path.join(test_path, f),
        target_size=(150,150)
    )

    x = image.img_to_array(img)
    x = x / 255.0
    x = np.expand_dims(x, axis=0)

    pred = model.predict(x, verbose=0)

    label = "Dog" if pred[0][0] > 0.5 else "Cat"

    plt.figure(figsize=(4,4))
    plt.imshow(img)
    plt.title(f"Prediction : {label}")
    plt.axis("off")
    plt.show()